In [24]:
# Import
import pandas as pd
import matplotlib.pyplot as plt

In [25]:
# Load Files
cagr = pd.read_csv("../data/processed/cagr_comparison_table.csv")

sharpe = pd.read_csv("../data/processed/sharpe_ratio_ranking.csv")

alpha = pd.read_csv("../data/processed/alpha_beta_analysis.csv")

drawdown = pd.read_csv("../data/processed/maximum_drawdown_analysis.csv")

performance = pd.read_csv("../data/processed/07_scheme_performance_clean.csv")

In [26]:
# Keep Required Columns
cagr = cagr[["amfi_code","CAGR_3Y"]]

sharpe = sharpe[["amfi_code","Sharpe_Ratio"]]

alpha = alpha[["amfi_code","Alpha"]]

drawdown = drawdown[["amfi_code","Maximum_Drawdown"]]

performance = performance[["amfi_code","expense_ratio_pct"]]

In [27]:
# Merge All Tables
score = cagr.merge(sharpe,on="amfi_code")

score = score.merge(alpha,on="amfi_code")

score = score.merge(drawdown,on="amfi_code")

score = score.merge(performance,on="amfi_code")

In [28]:
print(score.head())

   amfi_code    CAGR_3Y  Sharpe_Ratio     Alpha  Maximum_Drawdown  \
0     100016   1.292649     -0.201517  0.037476         -0.247344   
1     100025   3.916390     -0.567095  0.042818         -0.043083   
2     100033  32.442459      1.093699  0.271954         -0.162172   
3     101206  28.967695      1.027213  0.213998         -0.112916   
4     101207  -4.152381      0.162661  0.108971         -0.354469   

   expense_ratio_pct  
0               1.55  
1               0.56  
2               1.38  
3               1.60  
4               1.53  


In [29]:
# Create Ranks
## Higher value is better
score["Return_Rank"] = score["CAGR_3Y"].rank(ascending=False)

score["Sharpe_Rank"] = score["Sharpe_Ratio"].rank(ascending=False)

score["Alpha_Rank"] = score["Alpha"].rank(ascending=False)

In [30]:
## Lower Expense Ratio = Better
score["Expense_Rank"] = score["expense_ratio_pct"].rank(ascending=True)

In [31]:
## Lower Drawdown (less negative) = Better
score["Drawdown_Rank"] = score["Maximum_Drawdown"].rank(ascending=False)

In [32]:
# Convert Rank into Score
n = len(score)

score["Return_Score"] = ((n-score["Return_Rank"])/(n-1))*100

score["Sharpe_Score"] = ((n-score["Sharpe_Rank"])/(n-1))*100

score["Alpha_Score"] = ((n-score["Alpha_Rank"])/(n-1))*100

score["Expense_Score"] = ((n-score["Expense_Rank"])/(n-1))*100

score["Drawdown_Score"] = ((n-score["Drawdown_Rank"])/(n-1))*100

In [33]:
# Fund Score
score["Fund_Score"] = (

      score["Return_Score"]*0.30

    + score["Sharpe_Score"]*0.25

    + score["Alpha_Score"]*0.20

    + score["Expense_Score"]*0.15

    + score["Drawdown_Score"]*0.10

)

In [34]:
# Overall Rank
score = score.sort_values(
    by="Fund_Score",
    ascending=False
)

score["Overall_Rank"] = range(1,len(score)+1)

In [35]:
# Final Result
print(score[
[
"Overall_Rank",
"amfi_code",
"Fund_Score"
]
].head(10))

    Overall_Rank  amfi_code  Fund_Score
34             1     148567   85.897436
25             2     120505   81.794872
30             3     120843   81.538462
2              4     100033   80.256410
24             5     120504   79.487179
16             6     119094   76.410256
19             7     119551   74.166667
36             8     148569   73.012821
3              9     101206   67.371795
21            10     119598   66.538462


In [36]:
# Save CSV
score.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

print("Fund Scorecard saved successfully!")

Fund Scorecard saved successfully!
